# Day 10: Adam: A Method for Stochastic Optimization (PyTorch Implementation)

> **Bài báo gốc:** *Adam: A Method for Stochastic Optimization*  
> **Tác giả:** Diederik P. Kingma, Jimmy Lei Ba  
> **Xuất bản:** International Conference on Learning Representations (ICLR), 2015  
> **Tài liệu gốc:** [Day 10 Adam.pdf](./resources/Day%2010%20Adam.pdf)

---

## 📌 Giới Thiệu Chuyên Sâu Cho Người Mới Bắt Đầu

Trong hệ sinh thái **PyTorch**, `torch.optim.Adam` là bộ tối ưu hóa được sử dụng phổ biến nhất. Tuy nhiên, phần lớn kỹ sư chỉ sử dụng nó như một "hộp đen" mà chưa thấu hiểu:
1. Cách PyTorch quản lý các trạng thái nội bộ (`state dict`, `exp_avg`, `exp_avg_sq`).
2. Cách thức giải thuật thực thi các phép tính Tensor in-place (`mul_`, `addcmul_`, `addcdiv_`) để đạt tốc độ tối đa trên GPU/CUDA.
3. Tại sao công thức hiệu chỉnh độ chệch $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$ và $\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$ lại cứu nguy cho mạng nơ-ron không bị nổ gradient ở epoch đầu tiên.
4. Sự khác biệt cốt tử giữa **Adam** và **AdamW** (Decoupled Weight Decay).

Sổ tay này sẽ cùng bạn "mổ xẻ" toàn bộ kiến trúc trên!


In [ ]:
import os
import math
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.optim.optimizer import Optimizer
    HAS_TORCH = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[✓] Đã nạp thành công PyTorch phiên bản: {torch.__version__} | Thiết bị: {device}")
except ImportError:
    HAS_TORCH = False
    print("[!] Không tìm thấy PyTorch trong môi trường. Vui lòng cài đặt: pip install torch")


## 1. Giải Mã Cấu Trúc Bên Trong: Tự Xây Dựng `CustomAdam`

Một bộ tối ưu hóa trong PyTorch kế thừa từ `torch.optim.Optimizer`. Nó lưu trữ:
- `param_groups`: Danh sách các nhóm tham số (có thể có learning rate hoặc weight decay riêng).
- `state[p]`: Một từ điển lưu các vector trạng thái của từng tham số $p$:
  + `exp_avg`: Lưu mô-men bậc 1 $m_t$ (kỳ vọng gradient).
  + `exp_avg_sq`: Lưu mô-men bậc 2 $v_t$ (kỳ vọng bình phương gradient).
  + `step`: Bước đếm thời gian $t$ dùng để tính lũy thừa $\beta_1^t, \beta_2^t$ cho hiệu chỉnh độ chệch.


In [ ]:
if HAS_TORCH:
    class CustomAdam(Optimizer):
        def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
            defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
            super().__init__(params, defaults)

        @torch.no_grad()
        def step(self, closure=None):
            loss = None
            if closure is not None:
                with torch.enable_grad():
                    loss = closure()

            for group in self.param_groups:
                lr = group['lr']
                beta1, beta2 = group['betas']
                eps = group['eps']
                weight_decay = group['weight_decay']

                for p in group['params']:
                    if p.grad is None:
                        continue
                    grad = p.grad
                    state = self.state[p]

                    if len(state) == 0:
                        state['step'] = 0
                        state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                        state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                    exp_avg = state['exp_avg']
                    exp_avg_sq = state['exp_avg_sq']
                    state['step'] += 1
                    t = state['step']

                    # 1. L2 Regularization truyền thống
                    if weight_decay != 0:
                        grad = grad.add(p, alpha=weight_decay)

                    # 2. Cập nhật mô-men 1: m_t = beta1 * m_{t-1} + (1 - beta1) * g_t
                    exp_avg.mul_(beta1).add_(grad, alpha=1.0 - beta1)

                    # 3. Cập nhật mô-men 2: v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2
                    exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                    # 4. Hiệu chỉnh độ chệch
                    bias_correction1 = 1.0 - (beta1 ** t)
                    bias_correction2 = 1.0 - (beta2 ** t)
                    step_size = lr / bias_correction1
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

                    # 5. Cập nhật trọng số in-place
                    p.addcdiv_(exp_avg, denom, value=-step_size)

            return loss

    print("[✓] Đã biên dịch thành công CustomAdam!")


## 2. Thực Nghiệm 1: Khảo Sát Quỹ Đạo 2D Trên Địa Hình Dốc Lệch (Ravine)

Hàm mất mát dị hướng:
$$f(x, y) = 0.5 x^2 + 25.0 y^2$$
Điểm khởi hành: $(-4.0, 1.0)$. Cực tiểu toàn cục tại $(0.0, 0.0)$.
Ta so sánh trực quan quỹ đạo của 4 thuật toán: `SGD`, `SGD + Momentum`, `RMSprop`, và `Adam`.


In [ ]:
if HAS_TORCH:
    curvature_ratio = 50.0
    start_pt = (-4.0, 1.0)
    max_steps = 150

    opts_to_test = {
        "SGD (lr=0.035)": lambda p: optim.SGD([p], lr=0.035),
        "Momentum (lr=0.02)": lambda p: optim.SGD([p], lr=0.02, momentum=0.85),
        "RMSprop (lr=0.08)": lambda p: optim.RMSprop([p], lr=0.08, alpha=0.9),
        "Adam (lr=0.15)": lambda p: CustomAdam([p], lr=0.15, betas=(0.9, 0.999)),
    }

    trajectories = {}
    for name, opt_factory in opts_to_test.items():
        p = torch.tensor(list(start_pt), dtype=torch.float32, requires_grad=True)
        opt = opt_factory(p)
        traj = [p.detach().clone().numpy()]
        losses = []

        for _ in range(max_steps):
            opt.zero_grad()
            loss = 0.5 * (p[0] ** 2) + 0.5 * curvature_ratio * (p[1] ** 2)
            loss.backward()
            opt.step()
            traj.append(p.detach().clone().numpy())
            losses.append(loss.item())

        trajectories[name] = {"path": np.array(traj), "final_loss": losses[-1], "losses": losses}
        print(f"[*] {name:<20} | Loss cuối: {losses[-1]:.2e} | Vị trí: ({p[0].item():.2f}, {p[1].item():.2f})")


## 3. Thực Nghiệm 2: Huấn Luyện Mạng Nơ-ron Đa Tầng Bằng PyTorch

Ta tạo một bài toán phân loại nhị phân phi tuyến với 30 chiều đặc trưng (có độ thưa 40%), và huấn luyện mạng `DeepBenchmarkMLP` trong 60 epochs trên 5 bộ tối ưu hóa:
1. `Vanilla SGD`
2. `SGD + Momentum (0.9)`
3. `RMSprop (0.9)`
4. `Custom Adam (2014)`
5. `AdamW (2017)`


In [ ]:
if HAS_TORCH:
    class DeepBenchmarkMLP(nn.Module):
        def __init__(self, in_features=30, hidden_dims=[64, 32], num_classes=2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_features, hidden_dims[0]),
                nn.ReLU(),
                nn.Linear(hidden_dims[0], hidden_dims[1]),
                nn.ReLU(),
                nn.Linear(hidden_dims[1], num_classes)
            )

        def forward(self, x):
            return self.net(x)

    # Sinh dữ liệu
    np.random.seed(42)
    N, D = 500, 30
    X = np.random.randn(N, D).astype(np.float32)
    X[np.abs(X) < 0.5] = 0.0
    sig = 1.2 * X[:, 0] - 1.8 * (X[:, 1] ** 2) + 1.0 * np.sin(X[:, 2] * 2.0)
    y = (sig > 0).astype(np.int64)

    n_tr = 400
    X_train_t = torch.tensor(X[:n_tr]).to(device)
    y_train_t = torch.tensor(y[:n_tr]).to(device)
    X_test_t = torch.tensor(X[n_tr:]).to(device)
    y_test_t = torch.tensor(y[n_tr:]).to(device)

    epochs = 60
    base_lr = 0.01

    benchmark_opts = {
        "1. Vanilla SGD": lambda m: optim.SGD(m.parameters(), lr=base_lr),
        "2. SGD + Momentum": lambda m: optim.SGD(m.parameters(), lr=base_lr, momentum=0.9),
        "3. RMSprop": lambda m: optim.RMSprop(m.parameters(), lr=base_lr * 0.5, alpha=0.9),
        "4. Custom Adam": lambda m: CustomAdam(m.parameters(), lr=base_lr * 0.5),
        "5. AdamW": lambda m: optim.AdamW(m.parameters(), lr=base_lr * 0.5, weight_decay=0.01),
    }

    criterion = nn.CrossEntropyLoss()
    history = {k: {"train_loss": [], "test_acc": []} for k in benchmark_opts}

    print("=" * 75)
    print(f"{'Thuật Toán':<22} | {'Loss Cuối':<12} | {'Test Acc':<10}")
    print("-" * 75)

    for name, opt_fn in benchmark_opts.items():
        torch.manual_seed(100)
        m = DeepBenchmarkMLP().to(device)
        optimizer = opt_fn(m)

        for ep in range(epochs):
            m.train()
            optimizer.zero_grad()
            out = m(X_train_t)
            loss = criterion(out, y_train_t)
            loss.backward()
            optimizer.step()

            m.eval()
            with torch.no_grad():
                preds = torch.argmax(m(X_test_t), dim=1)
                acc = (preds == y_test_t).float().mean().item() * 100.0

            history[name]["train_loss"].append(loss.item())
            history[name]["test_acc"].append(acc)

        print(f"{name:<22} | {history[name]['train_loss'][-1]:<12.4f} | {history[name]['test_acc'][-1]:5.1f}%")
    print("=" * 75)


## 4. Phân Tích Hiệu Chỉnh Độ Chệch (Bias Correction) & Trực Quan Hóa

Tại sao hệ số hiệu chỉnh $1 / (1 - eta_1^t)$ và $1 / \sqrt{1 - eta_2^t}$ lại cần thiết?
- Ở bước $t = 1$, $eta_2 = 0.999 \implies 1 - eta_2^1 = 0.001 \implies 1 / \sqrt{1 - eta_2^1} = 31.62$.
- Nhờ hệ số nhân phóng đại này, mẫu số $\sqrt{v_1}$ được đưa về đúng quy mô thực tế của phương sai gradient!


In [ ]:
if HAS_TORCH:
    os.makedirs("outputs", exist_ok=True)
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    fig.suptitle("ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION (Kingma & Ba, ICLR 2015)\nPyTorch Academic Value Benchmark & Diagnostic Suite",
                 fontsize=14, fontweight="bold", y=0.98)

    # Panel 1: 2D Ravine Trajectories
    ax1 = axes[0, 0]
    X_grid, Y_grid = np.meshgrid(np.linspace(-4.5, 1.0, 200), np.linspace(-1.5, 1.5, 200))
    Z_grid = 0.5 * (X_grid ** 2) + 0.5 * 50.0 * (Y_grid ** 2)
    ax1.contour(X_grid, Y_grid, Z_grid, levels=np.logspace(-1, 3, 20), cmap="Blues", alpha=0.6)
    c_map = {"SGD (lr=0.035)": "red", "Momentum (lr=0.02)": "green", "RMSprop (lr=0.08)": "orange", "Adam (lr=0.15)": "blue"}
    for name, data in trajectories.items():
        path = data["path"]
        ax1.plot(path[:, 0], path[:, 1], '.-', color=c_map.get(name, "black"), label=name, markersize=3, linewidth=1.5, alpha=0.85)
    ax1.plot(0, 0, 'r*', markersize=14, label="Minimum (0, 0)")
    ax1.set_title("1. 2D Trajectories on Ill-Conditioned Ravine (50:1)", fontweight="bold")
    ax1.set_xlabel("x (Slow valley direction)")
    ax1.set_ylabel("y (Steep cliff direction)")
    ax1.legend(fontsize=8, loc="lower right")
    ax1.grid(True, linestyle="--", alpha=0.4)

    # Panel 2: Neural Net Training Loss
    ax2 = axes[0, 1]
    epochs_range = range(1, epochs + 1)
    plot_colors = ["#e74c3c", "#27ae60", "#f39c12", "#2980b9", "#8e44ad"]
    for idx, (name, data) in enumerate(history.items()):
        ax2.plot(epochs_range, data["train_loss"], label=name, color=plot_colors[idx % len(plot_colors)], linewidth=2)
    ax2.set_title("2. Neural Net Training Loss Convergence", fontweight="bold")
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Cross-Entropy Loss")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend(fontsize=8)

    # Panel 3: Bias Correction Multipliers
    ax3 = axes[1, 0]
    steps_arr = np.arange(1, 301)
    b1_curve = 1.0 / (1.0 - (0.9 ** steps_arr))
    b2_curve = 1.0 / np.sqrt(1.0 - (0.999 ** steps_arr))
    ax3.plot(steps_arr, b1_curve, 'b-', label=r"Moment 1 Scale: $1 / (1 - \beta_1^t)$", linewidth=2)
    ax3.plot(steps_arr, b2_curve, 'r-', label=r"Moment 2 Scale: $1 / \sqrt{1 - \beta_2^t}$", linewidth=2)
    ax3.set_title(r"3. Analytical Bias Correction Multipliers ($t = 1 \to 300$)", fontweight="bold")
    ax3.set_xlabel("Optimization Step (t)")
    ax3.set_ylabel("Magnification Multiplier")
    ax3.set_ylim(1.0, 12.0)
    ax3.grid(True, linestyle="--", alpha=0.5)
    ax3.legend(fontsize=9)

    # Panel 4: Test Accuracy Bar Chart
    ax4 = axes[1, 1]
    short_names = [k.split(". ")[-1] for k in history.keys()]
    acc_vals = [history[k]["test_acc"][-1] for k in history.keys()]
    bars = ax4.bar(short_names, acc_vals, color=plot_colors, alpha=0.85, width=0.5)
    ax4.set_title("4. Generalization Test Accuracy Comparison (%)", fontweight="bold")
    ax4.set_ylabel("Test Accuracy (%)")
    ax4.set_ylim(50, 100)
    ax4.grid(axis='y', linestyle="--", alpha=0.5)
    for bar, a in zip(bars, acc_vals):
        ax4.text(bar.get_x() + bar.get_width()/2.0, a + 1.0, f"{a:.1f}%", ha='center', va='bottom', fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.savefig("outputs/adam_academic_comparison.png", dpi=300)
    plt.show()
    print("[✓] Đã lưu thành công biểu đồ tại: outputs/adam_academic_comparison.png")


## 5. Tổng Kết Thực Chiến Cho Kỹ Sư Học Sâu

1. **Khi nào dùng Adam:** Dùng cho hầu hết mọi tác vụ khởi đầu (Computer Vision, NLP, Reinforcement Learning, Generative Models). Tốc độ hội tụ nhanh và ít phải chỉnh siêu tham số nhất.
2. **Khi nào dùng AdamW:** Luôn luôn dùng **AdamW** thay cho Adam nếu mô hình có sử dụng Weight Decay (đặc biệt là mô hình Transformer / LLM).
3. **Khi nào dùng SGD + Momentum:** Trong các bài toán phân loại ảnh với kiến trúc ResNet/ConvNeXt, khi bạn có đủ thời gian huấn luyện với Cosine Annealing Learning Rate Schedule, SGD + Momentum thường tìm được cực tiểu phẳng hơn (*flatter minima*), mang lại Test Accuracy cao hơn Adam từ 1-2%.
